# Module 8: Customer-Centric Personalization - The Archetype Lens

Rule-based behavioral archetypes for explaining how Chimera adapts recommendation composition across customer types.

## 1. Setup & Data Loading

In [1]:

from pathlib import Path
import sys
import textwrap

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PROCESSED = PROJECT_ROOT / 'data' / '02_processed'
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

from src import (
    ARCHETYPE_ORDER,
    assign_archetypes,
    build_archetype_case_study,
    build_temporal_holdout,
    compute_archetype_performance,
    compute_archetype_utility_profile,
    compute_household_features,
    evaluate_incremental_precision,
)

scored_candidates = pd.read_csv(DATA_PROCESSED / 'candidate_set_module3_scored.csv')
top5 = pd.read_csv(DATA_PROCESSED / 'top5_recommendations_module3.csv')
basket_diversity = pd.read_csv(DATA_PROCESSED / 'module6_basket_diversity.csv')
history = pd.read_parquet(DATA_PROCESSED / 'master_transactions.parquet')

split = build_temporal_holdout(history)
user_metrics = evaluate_incremental_precision(
    topk_frame=top5[top5['household_key'].isin(split.eligible_households)],
    train_items=split.train_items_by_user,
    test_items=split.test_items_by_user,
    top_k=5,
)

print(f'scored candidates: {len(scored_candidates):,}')
print(f'top-5 rows: {len(top5):,}')
print(f'households with holdout metrics: {len(user_metrics):,}')


C:\Users\hoduc\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scored candidates: 23,544
top-5 rows: 2,500
households with holdout metrics: 479


## 2. Archetype Definition (Step 8.1)

In [2]:

household_features = compute_household_features(scored_candidates, basket_diversity)
archetype_assignments = assign_archetypes(household_features)

population = (
    archetype_assignments.groupby('archetype', observed=False)
    .agg(
        households=('household_key', 'nunique'),
        avg_deal_sensitivity=('deal_sensitivity', 'mean'),
        avg_basket_diversity=('basket_diversity', 'mean'),
    )
    .reindex(ARCHETYPE_ORDER)
    .reset_index()
)

archetype_assignments.to_csv(DATA_PROCESSED / 'module8_archetype_assignments.csv', index=False)
population


,archetype,households,avg_deal_sensitivity,avg_basket_diversity
0,Routine Replenisher,180,0.756944,4.025560
1,Deal-Driven Explorer,177,0.921724,11.474364
2,Premium Discoverer,60,0.775684,9.348668
3,Frugal Loyalist,62,0.910254,4.853989


In [3]:

fig_scatter = px.scatter(
    archetype_assignments,
    x='deal_sensitivity',
    y='basket_diversity',
    color='archetype',
    category_orders={'archetype': ARCHETYPE_ORDER},
    hover_data=['household_key'],
    labels={'deal_sensitivity': 'Deal sensitivity', 'basket_diversity': 'Average basket diversity'},
    title='Customer archetypes by deal sensitivity and basket diversity',
)
fig_scatter.add_vline(x=float(archetype_assignments['deal_threshold'].iloc[0]), line_dash='dash', line_color='gray')
fig_scatter.add_hline(y=float(archetype_assignments['diversity_threshold'].iloc[0]), line_dash='dash', line_color='gray')
fig_scatter.update_layout(template='plotly_white', legend_title_text='Archetype')
fig_scatter.show()


## 3. Per-Archetype Recommendation Strategy (Step 8.2)

In [4]:

utility_profile = compute_archetype_utility_profile(top5, archetype_assignments)
utility_profile


,archetype,households,recommendations,avg_relevance,avg_uplift,avg_margin,avg_context,avg_utility,avg_relevance_contribution,avg_uplift_contribution,avg_margin_contribution,avg_context_contribution,source_pct_als,source_pct_mba,source_pct_both,source_pct_unknown
0,Routine Replenisher,180,900,0.927283,0.970290,0.986667,0.511111,0.887486,0.370913,0.242573,0.197333,0.076667,0.332222,0.368889,0.298889,0.0
1,Deal-Driven Explorer,177,885,0.903368,0.949956,0.984181,0.530395,0.875232,0.361347,0.237489,0.196836,0.079559,0.402260,0.415819,0.181921,0.0
2,Premium Discoverer,60,300,0.914834,0.968710,0.996667,0.519000,0.885294,0.365934,0.242178,0.199333,0.077850,0.376667,0.396667,0.226667,0.0
3,Frugal Loyalist,62,310,0.916057,0.959069,0.974194,0.510323,0.877577,0.366423,0.239767,0.194839,0.076548,0.361290,0.358065,0.280645,0.0


In [5]:

component_cols = ['avg_relevance', 'avg_uplift', 'avg_margin', 'avg_context']
component_labels = ['Relevance', 'Uplift', 'Margin', 'Context']
radar = go.Figure()
for _, row in utility_profile.iterrows():
    values = [row[col] for col in component_cols]
    radar.add_trace(
        go.Scatterpolar(
            r=values + [values[0]],
            theta=component_labels + [component_labels[0]],
            fill='toself',
            name=str(row['archetype']),
        )
    )
radar.update_layout(
    template='plotly_white',
    title='Average utility composition by archetype',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
)
radar.write_html(FIGURES / 'archetype_radar.html')
radar.show()


In [6]:

source_long = utility_profile.melt(
    id_vars='archetype',
    value_vars=['source_pct_als', 'source_pct_mba', 'source_pct_both'],
    var_name='source',
    value_name='share',
)
source_long['source'] = source_long['source'].str.replace('source_pct_', '', regex=False).str.upper()
fig_source = px.bar(
    source_long,
    x='archetype',
    y='share',
    color='source',
    category_orders={'archetype': ARCHETYPE_ORDER},
    labels={'share': 'Top-5 recommendation share', 'archetype': 'Archetype', 'source': 'Source'},
    title='Top-5 candidate source mix by archetype',
)
fig_source.update_layout(template='plotly_white', yaxis_tickformat='.0%')
fig_source.show()


## 4. Archetype-Specific Performance (Step 8.3)

In [7]:

performance = compute_archetype_performance(user_metrics, archetype_assignments)
performance


,archetype,households,avg_incremental_precision_at_5,avg_recommended_margin,avg_incremental_hits,precision_lift_vs_population,margin_lift_vs_population
0,Routine Replenisher,180,0.081111,0.986667,0.405556,0.163240,0.001299
1,Deal-Driven Explorer,177,0.053107,0.984181,0.265537,-0.238371,-0.001223
2,Premium Discoverer,60,0.100000,0.996667,0.500000,0.434132,0.011448
3,Frugal Loyalist,62,0.054839,0.974194,0.274194,-0.213541,-0.011359


In [8]:

performance_long = performance.melt(
    id_vars='archetype',
    value_vars=['avg_incremental_precision_at_5', 'avg_recommended_margin'],
    var_name='metric',
    value_name='value',
)
performance_long['metric'] = performance_long['metric'].map({
    'avg_incremental_precision_at_5': 'Incremental Precision@5',
    'avg_recommended_margin': 'Avg Recommended Margin',
})
fig_perf = px.bar(
    performance_long,
    x='archetype',
    y='value',
    color='metric',
    barmode='group',
    category_orders={'archetype': ARCHETYPE_ORDER},
    labels={'value': 'Metric value', 'archetype': 'Archetype', 'metric': 'Metric'},
    title='Archetype-specific recommendation performance',
)
fig_perf.update_layout(template='plotly_white')
fig_perf.write_html(FIGURES / 'archetype_performance_bar.html')
fig_perf.show()


## 5. Case Studies (Step 8.4)

In [9]:

case_studies = build_archetype_case_study(top5, archetype_assignments, history=history)
case_rows = []
for archetype, payload in case_studies.items():
    case_rows.append({
        'archetype': archetype,
        'household_key': payload.household_key,
        'deal_sensitivity': payload.deal_sensitivity,
        'basket_diversity': payload.basket_diversity,
        'narrative': payload.narrative,
    })
case_index = pd.DataFrame(case_rows)
case_index


D:\Project\Data_Driven_Marketing_complete-journey\src\archetypes.py:198: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  hh.groupby("COMMODITY_DESC", as_index=False)
D:\Project\Data_Driven_Marketing_complete-journey\src\archetypes.py:198: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  hh.groupby("COMMODITY_DESC", as_index=False)
D:\Project\Data_Driven_Marketing_complete-journey\src\archetypes.py:198: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and s

,archetype,household_key,deal_sensitivity,basket_diversity,narrative
0,Routine Replenisher,232,0.718153,5.547619,Household 232 represents the Routine Replenish...
1,Deal-Driven Explorer,164,0.904348,11.529412,Household 164 represents the Deal-Driven Explo...
2,Premium Discoverer,198,0.766376,9.621622,Household 198 represents the Premium Discovere...
3,Frugal Loyalist,296,0.916667,6.333333,Household 296 represents the Frugal Loyalist s...


In [10]:

def _draw_wrapped(ax, text, x, y, width=92, size=9, weight='normal'):
    wrapped = textwrap.fill(str(text), width=width)
    ax.text(x, y, wrapped, fontsize=size, fontweight=weight, va='top', ha='left', transform=ax.transAxes)

pdf_path = FIGURES / 'archetype_case_profiles.pdf'
with PdfPages(pdf_path) as pdf:
    for archetype in ARCHETYPE_ORDER:
        if archetype not in case_studies:
            continue
        payload = case_studies[archetype]
        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.axis('off')
        ax.text(0.04, 0.96, archetype, fontsize=18, fontweight='bold', va='top', transform=ax.transAxes)
        profile_text = (
            f"Representative household: {payload.household_key} | "
            f"deal sensitivity: {payload.deal_sensitivity:.2f} | "
            f"basket diversity: {payload.basket_diversity:.2f}"
        )
        ax.text(0.04, 0.90, profile_text, fontsize=10, va='top', transform=ax.transAxes)
        _draw_wrapped(ax, payload.narrative, 0.04, 0.85, width=118, size=10)

        rec_cols = [
            'COMMODITY_DESC', 'source_detail', 'Utility', 'Relevance', 'Uplift',
            'Normalized_Margin', 'Context', 'Relevance_Contribution',
            'Uplift_Contribution', 'Margin_Contribution', 'Context_Contribution'
        ]
        rec_table = payload.top_recommendations[rec_cols].copy()
        rec_table.insert(0, 'Rank', range(1, len(rec_table) + 1))
        rec_table = rec_table.rename(columns={'COMMODITY_DESC': 'Item', 'source_detail': 'Source', 'Normalized_Margin': 'Margin'})
        for col in rec_table.select_dtypes(include='number').columns:
            if col != 'Rank':
                rec_table[col] = rec_table[col].map(lambda value: f'{value:.3f}')
        rec_table['Item'] = rec_table['Item'].astype(str).str.slice(0, 24)

        ax.text(0.04, 0.72, 'Top-5 recommendations and utility decomposition', fontsize=12, fontweight='bold', va='top', transform=ax.transAxes)
        table = ax.table(
            cellText=rec_table.values,
            colLabels=rec_table.columns,
            loc='upper left',
            bbox=[0.04, 0.36, 0.92, 0.31],
            cellLoc='left',
        )
        table.auto_set_font_size(False)
        table.set_fontsize(7)

        history_table = payload.purchase_history_summary.copy()
        ax.text(0.04, 0.29, 'Recent purchase-history profile', fontsize=12, fontweight='bold', va='top', transform=ax.transAxes)
        if history_table.empty:
            _draw_wrapped(ax, 'No purchase history rows available for this household.', 0.04, 0.24, size=9)
        else:
            history_table = history_table.rename(columns={'COMMODITY_DESC': 'Item'})
            history_table['Item'] = history_table['Item'].astype(str).str.slice(0, 36)
            for col in ['revenue']:
                if col in history_table.columns:
                    history_table[col] = history_table[col].map(lambda value: f'{value:.2f}')
            htable = ax.table(
                cellText=history_table.values,
                colLabels=history_table.columns,
                loc='upper left',
                bbox=[0.04, 0.05, 0.62, 0.19],
                cellLoc='left',
            )
            htable.auto_set_font_size(False)
            htable.set_fontsize(8)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
print(f'Saved {pdf_path}')


Saved D:\Project\Data_Driven_Marketing_complete-journey\reports\figures\archetype_case_profiles.pdf


## 6. Summary & Key Findings

In [11]:

summary = (
    population.merge(utility_profile, on='archetype', how='left', suffixes=('', '_profile'))
    .merge(performance, on='archetype', how='left', suffixes=('', '_performance'))
)
summary.to_csv(DATA_PROCESSED / 'module8_archetype_summary.csv', index=False)

assert len(archetype_assignments) == len(household_features)
assert archetype_assignments['archetype'].nunique() == 4
assert (DATA_PROCESSED / 'module8_archetype_assignments.csv').exists()
assert (DATA_PROCESSED / 'module8_archetype_summary.csv').exists()
assert (FIGURES / 'archetype_radar.html').exists()
assert (FIGURES / 'archetype_performance_bar.html').exists()
assert (FIGURES / 'archetype_case_profiles.pdf').exists()

summary


,archetype,households,avg_deal_sensitivity,avg_basket_diversity,households_profile,recommendations,avg_relevance,avg_uplift,avg_margin,avg_context,...,source_pct_als,source_pct_mba,source_pct_both,source_pct_unknown,households_performance,avg_incremental_precision_at_5,avg_recommended_margin,avg_incremental_hits,precision_lift_vs_population,margin_lift_vs_population
0,Routine Replenisher,180,0.756944,4.025560,180,900,0.927283,0.970290,0.986667,0.511111,...,0.332222,0.368889,0.298889,0.0,180,0.081111,0.986667,0.405556,0.163240,0.001299
1,Deal-Driven Explorer,177,0.921724,11.474364,177,885,0.903368,0.949956,0.984181,0.530395,...,0.402260,0.415819,0.181921,0.0,177,0.053107,0.984181,0.265537,-0.238371,-0.001223
2,Premium Discoverer,60,0.775684,9.348668,60,300,0.914834,0.968710,0.996667,0.519000,...,0.376667,0.396667,0.226667,0.0,60,0.100000,0.996667,0.500000,0.434132,0.011448
3,Frugal Loyalist,62,0.910254,4.853989,62,310,0.916057,0.959069,0.974194,0.510323,...,0.361290,0.358065,0.280645,0.0,62,0.054839,0.974194,0.274194,-0.213541,-0.011359
